In [3]:
import pandas as pd
import numpy as np
import json

df = pd.read_csv("../afl_players_round_by_round_stats_raw.csv", low_memory=False)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nYear range:", df['year'].min(), "-", df['year'].max())
print("Unique rounds:", sorted(df['round'].unique()))


Shape: (274089, 36)
Columns: ['id', 'team', 'year', 'career_game_count', 'opponent', 'round', 'result', 'jersey_num', 'kicks', 'marks', 'handballs', 'disposals', 'goals', 'behinds', 'hit_outs', 'tackles', 'rebound_50s', 'inside_50s', 'clearances', 'clangers', 'free_kicks_for', 'free_kicks_against', 'brownlow_votes', 'contested_possessions', 'uncontested_possessions', 'contested_marks', 'marks_inside_50', 'one_percenters', 'bounces', 'goal_assist', 'percentage_of_game_played', 'player_id', 'match_date', 'fantasy_points', 'score', 'margin']

Year range: 1983 - 2025
Unique rounds: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '3', '4', '5', '6', '7', '8', '9', 'EF', 'GF', 'PF', 'QF', 'SF']


In [4]:
dqa = {}

dqa['total_rows'] = len(df)
dqa['full_duplicate_rows'] = int(df.duplicated().sum())
dqa['duplicate_keys_player_year_round_team'] = int(df.duplicated(subset=['player_id','year','round','team']).sum())
dqa['negative_disposals'] = int((df['disposals'] < 0).sum())
dqa['negative_fantasy_points'] = int((df['fantasy_points'] < 0).sum())
dqa['round_0_preseason_rows'] = int((df['round'] == '0').sum())
dqa['finals_rows'] = int(df['round'].isin(['EF','QF','SF','PF','GF']).sum())
dqa['team_name_variants'] = int(df['team'].nunique())
dqa['null_player_id'] = int(df['player_id'].isna().sum())
dqa['unparseable_dates'] = int(pd.to_datetime(df['match_date'], errors='coerce').isna().sum())

print("Data Quality Assessment (raw):")
for k, v in dqa.items():
    print(f"  {k}: {v}")


Data Quality Assessment (raw):
  total_rows: 274089
  full_duplicate_rows: 10
  duplicate_keys_player_year_round_team: 103
  negative_disposals: 723
  negative_fantasy_points: 167
  round_0_preseason_rows: 276
  finals_rows: 12802
  team_name_variants: 20
  null_player_id: 0
  unparseable_dates: 0


In [5]:
rnd = df.copy()
rows_before = len(rnd)

n = int(rnd.duplicated().sum())
rnd = rnd.drop_duplicates()
print(f"Fix 1:  Dropped {n} exact duplicate rows")


Fix 1:  Dropped 10 exact duplicate rows


In [7]:
# Fix 2: non-exact duplicate keys — keep both (legitimate separate matches) ──
# Investigation showed two causes:
# a) 2010 GF replay (Sep 25 draw + Oct 2 replay) — both games are real
# b) 2025 split-round — Gold Coast played two matches in the same round number
# Both are legitimate and should be kept. The 'id' column differentiates them.
non_exact_dups = rnd[rnd.duplicated(subset=['player_id','year','round','team'], keep=False) &
                     ~rnd.duplicated(keep=False)]
n = len(non_exact_dups)

print(f"Fix 2: Retained {n} rows with duplicate keys (legitimate separate matches)")
print("  Sample:", rnd[(rnd['year']==2010) & (rnd['round']=='GF')]['match_date'].unique())


Fix 2: Retained 186 rows with duplicate keys (legitimate separate matches)
  Sample: ['2010-09-25' '2010-10-02']


In [9]:
#Fix 3: negative disposals — drop
n = int((rnd['disposals'] < 0).sum())
rnd = rnd[rnd['disposals'].isna() | (rnd['disposals'] >= 0)].copy()
print(f"Fix 3: Dropped {n} rows with negative disposals")


Fix 3: Dropped 0 rows with negative disposals


In [11]:
# Fix 4: negative fantasy_points — keep
n = int((rnd['fantasy_points'] < 0).sum())
print(f"Fix 4: Retained {n} rows with negative fantasy_points (legitimate)")


Fix 4: Retained 161 rows with negative fantasy_points (legitimate)


In [12]:
#Fix 5: round = '0' preseason games — drop
n = int((rnd['round'] == '0').sum())
rnd = rnd[rnd['round'] != '0'].copy()

print(f"Fix 5: Dropped {n} preseason rows (round = 0)")


Fix 5: Dropped 275 preseason rows (round = 0)


In [15]:
# Fix 7: parse match_date to datetime
rnd['match_date'] = pd.to_datetime(rnd['match_date'], errors='coerce')
n_bad = int(rnd['match_date'].isna().sum())
print(f"Fix 7: Parsed match_date to datetime. Unparseable: {n_bad}")


Fix 7: Parsed match_date to datetime. Unparseable: 0


In [17]:
# Fix 8: convert round to numeric where possible, keep finals as text
# Create a separate numeric round column for ordering/plotting
def round_to_num(r):
    try:
        return int(r)
    except:
        return None

rnd['round_num'] = rnd['round'].apply(round_to_num)

print("Fix 8: Created round_num column")
print("round_num nulls (finals):", rnd['round_num'].isna().sum())


Fix 8: Created round_num column
round_num nulls (finals): 12778


In [18]:
rnd.to_csv("round_by_round_cleaned.csv", index=False)

print("Saved:")
print("  round_by_round_cleaned.csv")
print(f"\nFinal shape: {rnd.shape}")


Saved:
  round_by_round_cleaned.csv

Final shape: (273082, 37)
